# Pipeline de prediction de charge : Energy Informatics 2
### Jean Varone, HES-SO Valais-Wallis
### Collaboration OIKEN, Mars 2026


**Architecture du pipeline** : `config` → `acquisition` → `normalization` → `features`

## 1. Configuration (`config.py`)

Centralise tous les parametres du projet : chemins, credentials InfluxDB, liste des sites meteo et des mesures.

In [ ]:
# Extrait de config.py

from pathlib import Path

# Chemins
PROJECT_ROOT = Path(".")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# 4 stations MeteoSuisse couvrant le Valais central
SITES = ["Sion", "Visp", "Montana", "Evolene / Villa"]

# 9 mesures reelles x 4 stations (agregees en moyenne : 9 colonnes)
# 8 variables de prevision x 4 variantes (ctrl, q10, q90, stde) x 4 stations (agregees : 32 colonnes)
MEASUREMENTS_REAL = [
    "Air temperature 2m above ground (current value)",
    "Global radiation (ten minutes mean)",
    "Precipitation (ten minutes total)",
    # + 6 autres : pression, humidite, vent, ensoleillement, rafales, direction du vent
]

# Periode : oct. 2022 - sept. 2025 (~3 ans)
OIKEN_START = "2022-10-01"
OIKEN_END   = "2025-09-30"
TIMEZONE    = "Europe/Zurich"

print(f"Sites meteo : {SITES}")
print(f"Periode : {OIKEN_START} -> {OIKEN_END}")
print(f"Mesures reelles : 9 variables x 4 sites (moyennees -> 9 colonnes)")
print(f"Previsions PRED : 8 variables x 4 variantes = 32 colonnes")

## 2. Acquisition (`acquisition.py`)

Deux sources de donnees :

| Source | Format | Contenu |
|--------|--------|---------|
| **OIKEN** | CSV | Charge standardisee, forecast baseline, production PV (4 zones) |
| **MeteoSuisse** | InfluxDB (HES-SO) | Mesures reelles (10 min) + previsions COSMO (3h) |

Traitements cles :
- Detection auto du separateur CSV (`;` ou `,`) et des decimales `,` → `.`
- Gestion des `#N/A` → null → interpolation
- Sauvegarde en **Parquet** : lecture ~10x plus rapide que CSV, typage conserve

In [ ]:
# Extrait simplifie de acquisition.py

# 1) Chargement CSV OIKEN
#    Parsing robuste : separateur auto, decimales FR, dates DD.MM.YYYY
#    Calcul de pv_total = somme des 4 zones PV

# 2) Extraction InfluxDB (MeteoSuisse)
#    Requetes Flux : filtre par measurement + site + prediction run
#    pivot() : format long (1 ligne par mesure) -> large (1 colonne par variable)
#    Run "00" (minuit) pour les previsions : couvre J+1

# Resultat : 3 fichiers Parquet
for f in ["oiken_clean.parquet", "meteo_real.parquet", "meteo_pred.parquet"]:
    print(f"  -> data/raw/{f}")

## 3. Normalisation (`normalization.py`)

Aligne toutes les sources sur une grille temporelle commune : **UTC, pas de 15 min**.

Problemes resolus :
- **Changement d'heure (DST)** : 3 timestamps inexistants (printemps) supprimes, 12 doublons (automne) dedupliques
- **Reechantillonnage** : MeteoSuisse 10 min → 15 min, previsions 3h → 15 min (interpolation lineaire)
- **Agregation multi-sites** : moyenne des 4 stations valaisannes

In [ ]:
# Logique cle de normalization.py

# Conversion DST -> UTC (OIKEN est en heure locale)
# pl.col("timestamp")
#   .dt.replace_time_zone("Europe/Zurich", ambiguous="latest", non_existent="null")
#   .dt.convert_time_zone("UTC")

# Reechantillonnage sur grille reguliere 15 min :
# 1. Creer grille : pl.datetime_range(min, max, interval="15m")
# 2. Left join avec les donnees
# 3. Interpolation lineaire des trous

# Fusion finale :
# oiken LEFT JOIN meteo_real LEFT JOIN meteo_pred ON timestamp

print("Dataset fusionne : ~105'105 lignes x (charge + meteo reelle + previsions)")
print("  3 lignes supprimees (DST printemps)")
print("  12 doublons resolus (DST automne)")
print("  12 #N/A interpoles dans forecast_load")

## 4. Feature Engineering (`features.py`)

Transforme le dataset brut en features exploitables par le modele.

| Categorie | Features | Justification |
|-----------|----------|---------------|
| **Calendaire** | hour/weekday/month sin+cos, is_weekend, is_holiday, is_school_holiday | Profil journalier/hebdo/saisonnier marque |
| **Lags charge** | load_lag_96 (J-1), load_lag_672 (J-7) | Memoire temporelle, disponible en J+1 |
| **Lags PV** | pv_total_lag_96, pv_total_lag_672 | Production PV passee |
| **Meteo PRED** | 32 variables (8 var x 4 variantes) | Seule meteo disponible en J+1 |
| **Interactions** | temp x hour, radiation x elevation, uncertainty ranges | Capture des non-linearites |
| **Position solaire** | solar_elevation (pvlib) | Proxy de l'ensoleillement potentiel |

**Contrainte J+1** : pas de lags courts (lag_1, lag_4) ni de meteo reelle → data leakage interdit.

In [ ]:
# Encodage cyclique : evite les discontinuites (23h -> 0h)
import numpy as np

hour = np.arange(24)
hour_sin = np.sin(2 * np.pi * hour / 24)
hour_cos = np.cos(2 * np.pi * hour / 24)

print("Encodage cyclique de l'heure :")
print(f"  0h  : sin={hour_sin[0]:.2f}, cos={hour_cos[0]:.2f}")
print(f"  6h  : sin={hour_sin[6]:.2f}, cos={hour_cos[6]:.2f}")
print(f"  12h : sin={hour_sin[12]:.2f}, cos={hour_cos[12]:.2f}")
print(f"  23h : sin={hour_sin[23]:.2f}, cos={hour_cos[23]:.2f}")
print()
print("Continuite preservee : 23h est proche de 0h dans l'espace (sin, cos)")
print()
print("Dataset final : ~104'433 lignes x ~95 features")
print("  (672 lignes perdues = lag max de 7 jours x 96 pas/jour)")

## 5. Visualisations

Justification visuelle des choix de features.

In [ ]:
import sys
from pathlib import Path

# Ajouter la racine du projet au path (pour que 'from src...' fonctionne)
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

TIMEZONE = "Europe/Zurich"

%matplotlib inline
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
BLUE, RED, GREEN, ORANGE = "#2b6cb0", "#e53e3e", "#38a169", "#dd6b20"

# Charger le dataset directement depuis le Parquet
data_processed = project_root / "data" / "processed"
feat_path = data_processed / "dataset_features.parquet"

if feat_path.exists():
    df = pl.read_parquet(feat_path)
    print(f"Dataset charge : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
else:
    print(f"ERREUR : {feat_path} introuvable.")
    print("Lancer d'abord dans le terminal :")
    print("  .venv\\Scripts\\python.exe -m src.normalization")
    print("  .venv\\Scripts\\python.exe -m src.features")

### 5.1 Correlations des features J+1 avec la charge

In [ ]:
j1_features = [
    "load_lag_96", "load_lag_672", "pv_total_lag_96", "pv_total_lag_672",
    "pred_t_2m_ctrl", "pred_glob_ctrl", "pred_t_2m_stde", "pred_glob_stde",
    "doy_cos", "month_cos",
]
available = [f for f in j1_features if f in df.columns]
correlations = {}
for col in available:
    corr = df.select(pl.corr("load", col)).item()
    if corr is not None and not np.isnan(corr):
        correlations[col] = corr

sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
names = [x[0] for x in sorted_corr][::-1]
values = [x[1] for x in sorted_corr][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(names)), values, color=[BLUE if v > 0 else RED for v in values], alpha=0.8)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel("Correlation avec la charge (load)")
ax.set_title("Features utilisees en forecast J+1", fontweight="bold")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

### 5.2 Data leakage des lags courts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if "load_lag_1" in df.columns:
    sample = df.sample(n=min(5000, df.shape[0]), seed=42)
    ax = axes[0]
    ax.scatter(sample["load_lag_1"].to_numpy(), sample["load"].to_numpy(),
               alpha=0.1, s=5, color=RED)
    ax.plot([0, 2], [0, 2], "k--", linewidth=1, label="y = x")
    ax.set_xlabel("load_lag_1 (charge il y a 15 min)")
    ax.set_ylabel("load (charge actuelle)")
    ax.set_title("load_lag_1 vs load : quasi identiques\n-> DATA LEAKAGE en J+1",
                 fontweight="bold", color=RED)
    corr = df.select(pl.corr("load", "load_lag_1")).item()
    ax.text(0.05, 0.95, f"r = {corr:.4f}", transform=ax.transAxes,
            fontsize=12, fontweight="bold", va="top", color=RED)
    ax.legend()

ax = axes[1]
n = df.shape[0]
slice_df = df.slice(n // 2, 192)
ts = slice_df["timestamp"].to_numpy()
ax.plot(ts, slice_df["load"].to_numpy(), label="load (reel)", color=BLUE, linewidth=1.5)
if "load_lag_1" in slice_df.columns:
    ax.plot(ts, slice_df["load_lag_1"].to_numpy(), label="load_lag_1",
            color=RED, linewidth=1, linestyle="--", alpha=0.7)
if "load_lag_96" in slice_df.columns:
    ax.plot(ts, slice_df["load_lag_96"].to_numpy(), label="load_lag_96 (J-1)",
            color=GREEN, linewidth=1, linestyle=":", alpha=0.7)
ax.set_xlabel("Temps")
ax.set_ylabel("Charge normalisee")
ax.set_title("Comparaison des lags sur 2 jours", fontweight="bold")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m %Hh"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 5.3 Patterns temporels de la charge

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
local_ts = df["timestamp"].dt.convert_time_zone(TIMEZONE)

ax = axes[0]
hour_data = df.with_columns(local_ts.dt.hour().alias("h"))
hourly = hour_data.group_by("h").agg(
    pl.col("load").mean().alias("mean"), pl.col("load").std().alias("std")
).sort("h")
h = hourly["h"].to_numpy(); m = hourly["mean"].to_numpy(); s = hourly["std"].to_numpy()
ax.plot(h, m, color=BLUE, linewidth=2)
ax.fill_between(h, m - s, m + s, alpha=0.2, color=BLUE)
ax.set_xlabel("Heure"); ax.set_ylabel("Charge moyenne")
ax.set_title("Profil journalier\n-> justifie hour_sin / hour_cos", fontweight="bold")
ax.set_xticks(range(0, 24, 3))

ax = axes[1]
wd_data = df.with_columns(local_ts.dt.weekday().alias("wd"))
weekly = wd_data.group_by("wd").agg(pl.col("load").mean()).sort("wd")
days = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
ax.bar(range(7), weekly["load"].to_numpy(), color=BLUE, alpha=0.8)
ax.set_xticks(range(7)); ax.set_xticklabels(days)
ax.set_ylabel("Charge moyenne")
ax.set_title("Profil hebdomadaire\n-> justifie weekday_sin + is_weekend", fontweight="bold")

ax = axes[2]
mo_data = df.with_columns(local_ts.dt.month().alias("mo"))
monthly = mo_data.group_by("mo").agg(pl.col("load").mean()).sort("mo")
months = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
ax.bar(range(12), monthly["load"].to_numpy(), color=BLUE, alpha=0.8)
ax.set_xticks(range(12)); ax.set_xticklabels(months)
ax.set_ylabel("Charge moyenne")
ax.set_title("Profil saisonnier\n-> justifie month_sin + doy_sin", fontweight="bold")

plt.tight_layout()
plt.show()

### 5.4 Temperature vs charge

In [ ]:
if "pred_t_2m_ctrl" in df.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    sample = df.sample(n=min(10000, df.shape[0]), seed=42)
    ax.scatter(sample["pred_t_2m_ctrl"].to_numpy(), sample["load"].to_numpy(),
               alpha=0.05, s=3, color=BLUE)

    temp = df["pred_t_2m_ctrl"].to_numpy()
    load = df["load"].to_numpy()
    bins = np.linspace(np.nanpercentile(temp, 1), np.nanpercentile(temp, 99), 30)
    bin_centers, bin_means = [], []
    for i in range(len(bins) - 1):
        mask = (temp >= bins[i]) & (temp < bins[i+1])
        if mask.sum() > 50:
            bin_centers.append((bins[i] + bins[i+1]) / 2)
            bin_means.append(np.mean(load[mask]))

    ax.plot(bin_centers, bin_means, color=RED, linewidth=2.5, label="Moyenne par bin")
    ax.set_xlabel("Temperature predite (C)"); ax.set_ylabel("Charge normalisee")
    ax.set_title("Temperature vs Charge\nLe chauffage domine la consommation", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("pred_t_2m_ctrl absent du dataset")

### 5.5 Effet vacances et jours feries

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

normal = df.filter(
    (pl.col("is_weekend") == 0) & (pl.col("is_holiday") == 0) & (pl.col("is_school_holiday") == 0)
)["load"].to_numpy()
weekend = df.filter(pl.col("is_weekend") == 1)["load"].to_numpy()
vacances = df.filter(
    (pl.col("is_school_holiday") == 1) & (pl.col("is_weekend") == 0)
)["load"].to_numpy()
feries = df.filter(
    (pl.col("is_holiday") == 1) & (pl.col("is_weekend") == 0)
)["load"].to_numpy()

bp = ax.boxplot([normal.tolist(), weekend.tolist(), vacances.tolist(), feries.tolist()],
                labels=["Jour ouvre", "Week-end", "Vacances\nscolaires", "Jour ferie"],
                patch_artist=True, showfliers=False, medianprops=dict(color="black", linewidth=2))

for patch, color in zip(bp["boxes"], [BLUE, ORANGE, GREEN, RED]):
    patch.set_facecolor(color); patch.set_alpha(0.6)

ax.set_ylabel("Charge normalisee")
ax.set_title("Impact du type de jour sur la charge\n-> justifie is_weekend, is_holiday, is_school_holiday",
             fontweight="bold")
plt.tight_layout()
plt.show()